# When to Stop a Trading System — Live Monitoring Framework

This notebook puts together the full framework in one place: a synthetic
walk-forward return series acts as the benchmark, and a synthetic live
series (with a deliberately engineered alpha decay halfway through) is
checked against it with every method discussed:

1. Generate historical (walk-forward) and live returns
2. Max drawdown curve from Monte Carlo vs. live drawdown
3. Monte Carlo equity bands vs. live equity
4. Percentile ranking of monthly returns
5. CUSUM
6. Rolling alpha regression

All thresholds are calibrated **once**, from the historical series only,
before ever looking at the live data — exactly the "decided a priori"
principle the whole framework rests on.

In [66]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go
from arch.bootstrap import optimal_block_length
from scipy.stats import norm

pd.set_option("display.precision", 4)
TEMPLATE = "plotly_dark"

## 1. Generate historical and live returns

Historical = walk-forward out-of-sample trade returns (the benchmark).
Live = a shorter series that starts with the *same* edge, then switches
to a clearly worse distribution partway through — a stand-in for real
alpha decay, so every method below has something genuine to catch.

In [67]:
rng = np.random.default_rng(42)

TRADES_PER_YEAR = 250          # used both for date spacing and window sizing later
N_HIST = 600
N_HEALTHY = 300                # live trades matching the historical edge
N_DECAY = 50                   # live trades with the engineered decay
N_LIVE = N_HEALTHY + N_DECAY
GAP_DAYS = 365.25 / TRADES_PER_YEAR

# historical (walk-forward OOS) returns
hist_dates = pd.Timestamp("2023-01-01") + pd.to_timedelta(np.arange(N_HIST) * GAP_DAYS, unit="D")
hist_returns = rng.normal(0.0025, 0.009, N_HIST)
hist_df = pd.DataFrame({"date": hist_dates, "return": hist_returns})

# live returns: healthy phase, then engineered decay (lower mean, higher vol)
live_start = hist_dates[-1] + pd.Timedelta(days=30)
live_dates = live_start + pd.to_timedelta(np.arange(N_LIVE) * GAP_DAYS, unit="D")
live_returns = np.concatenate([
    rng.normal(0.0025, 0.009, N_HEALTHY),   # same edge as historical
    rng.normal(-0.0060, 0.010, N_DECAY),    # engineered decay starts here
])
live_df = pd.DataFrame({"date": live_dates, "return": live_returns})

print(f"historical: {len(hist_df)} trades, {hist_df['date'].min().date()} -> {hist_df['date'].max().date()}")
print(f"live:       {len(live_df)} trades, {live_df['date'].min().date()} -> {live_df['date'].max().date()}")
print(f"decay engineered from live trade #{N_HEALTHY} onward")

historical: 600 trades, 2023-01-01 -> 2025-05-25
live:       350 trades, 2025-06-24 -> 2026-11-16
decay engineered from live trade #300 onward


In [68]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=(1 + hist_df["return"]).cumprod(), mode="lines", name="historical (WF)"))
fig.add_trace(go.Scatter(y=(1 + live_df["return"]).cumprod(), mode="lines", name="live"))
fig.add_vline(x=N_HEALTHY, line_dash="dot", line_color="gray", annotation_text="decay starts here")
fig.update_layout(title="Cumulative return — historical vs. live", template=TEMPLATE,
                   xaxis_title="trade #", yaxis_title="cumulative return", height = 1000)
fig.show()

## 2. Max drawdown curve from Monte Carlo vs. live drawdown

The key idea: the max-drawdown threshold is **not a single number** — it
grows with the number of trades, because a deeper worst-case drawdown
becomes statistically unsurprising the longer a system runs. What is
fixed in advance is the whole *threshold curve*, calibrated once from
the historical series via **block bootstrap** (to preserve whatever
autocorrelation is actually in the data, instead of assuming trades are
independent). Block length is picked automatically (Politis–White),
not guessed.

In [69]:
def block_bootstrap_paths(returns, n_sims, horizon, block_size, seed=0):
    """Simulate `n_sims` paths of length `horizon` by resampling blocks of `returns`."""
    rng = np.random.default_rng(seed)
    returns = np.asarray(returns)
    n = len(returns)
    paths = np.empty((n_sims, horizon))
    for i in range(n_sims):
        blocks = []
        while sum(len(b) for b in blocks) < horizon:
            start = rng.integers(0, n - block_size + 1)
            blocks.append(returns[start:start + block_size])
        paths[i, :] = np.concatenate(blocks)[:horizon]
    return paths


block_size = int(round(optimal_block_length(hist_df["return"])["stationary"].iloc[0]))
print("optimal block size (Politis-White):", block_size)

N_SIMS = 3000
sim_returns = block_bootstrap_paths(hist_df["return"].values, N_SIMS, N_LIVE, block_size, seed=1)
sim_equity = np.cumprod(1 + sim_returns, axis=1)   # shape (N_SIMS, N_LIVE), reused in section 3 too

optimal block size (Politis-White): 2


In [70]:
# worst drawdown reached so far along each simulated path -> 5th percentile curve
sim_running_max = np.maximum.accumulate(sim_equity, axis=1)
sim_dd = sim_equity / sim_running_max - 1
sim_worst_dd = np.minimum.accumulate(sim_dd, axis=1)
dd_threshold = np.quantile(sim_worst_dd, 0.05, axis=0)     # one value per trade #, fixed in advance

# same transformation applied to the live equity curve
live_equity = (1 + live_df["return"]).cumprod().values
live_running_max = np.maximum.accumulate(live_equity)
live_dd = live_equity / live_running_max - 1
live_worst_dd = np.minimum.accumulate(live_dd)

breach = live_worst_dd < dd_threshold
first_breach = np.argmax(breach) if breach.any() else None
print("drawdown breach starts at live trade #:", first_breach)

drawdown breach starts at live trade #: 309


In [71]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=dd_threshold * 100, mode="lines", name="5th pct. threshold (pre-committed)",
                          line=dict(color="gray", dash="dash")))
fig.add_trace(go.Scatter(y=live_worst_dd * 100, mode="lines", name="live worst drawdown",
                          line=dict(color="#EF553B")))
if breach.any():
    idx = np.where(breach)[0]
    fig.add_trace(go.Scatter(x=idx, y=live_worst_dd[idx] * 100, mode="markers",
                              name="breach", marker=dict(color="red", size=7)))
fig.update_layout(title="Max drawdown: live vs. bootstrap threshold curve", template=TEMPLATE,
                   xaxis_title="trade #", yaxis_title="drawdown (%)", height = 1000)
fig.show()

## 3. Monte Carlo equity bands vs. live equity

Same simulated paths as section 2, this time turned into percentile
bands on the equity level itself (5th / 50th / 95th) instead of on the
drawdown. Useful as a first visual check, but note it answers a
slightly different question than the drawdown test above — a path can
sit inside the equity band while still having crossed a drawdown
threshold earlier in its history, so the two checks are complementary,
not interchangeable.

In [72]:
mc_lower = np.quantile(sim_equity, 0.05, axis=0)
mc_median = np.quantile(sim_equity, 0.50, axis=0)
mc_upper = np.quantile(sim_equity, 0.95, axis=0)

fig = go.Figure()
fig.add_trace(go.Scatter(y=mc_upper, mode="lines", line=dict(color="rgba(150,120,255,0.4)"), name="95th pct."))
fig.add_trace(go.Scatter(y=mc_lower, mode="lines", line=dict(color="rgba(150,120,255,0.4)"), name="5th pct.",
                          fill="tonexty", fillcolor="rgba(150,120,255,0.12)"))
fig.add_trace(go.Scatter(y=mc_median, mode="lines", line=dict(color="rgba(150,120,255,0.7)", dash="dot"), name="median"))
fig.add_trace(go.Scatter(y=live_equity, mode="lines", line=dict(color="#00CC96", width=3), name="live equity"))
fig.update_layout(title="Live equity vs. Monte Carlo bands", template=TEMPLATE,
                   xaxis_title="trade #", yaxis_title="equity (x initial capital)", height = 1000)
fig.show()

## 4. Percentile ranking of monthly returns

Trade returns are compounded within each calendar month to get one
number per month, both for the historical (walk-forward) series and
for live. Each live month is then percentile-ranked against the
historical monthly-return distribution — a single bad month is only
informative relative to how bad historical months have actually been,
not against some arbitrary fixed bar.

In [73]:
def monthly_compounded(df):
    return df.set_index("date")["return"].resample("ME").apply(lambda r: (1 + r).prod() - 1)

hist_monthly = monthly_compounded(hist_df)
live_monthly = monthly_compounded(live_df)

pct_ranks = live_monthly.apply(lambda r: (hist_monthly < r).mean() * 100)
pct_ranks.name = "percentile_vs_historical"
print(pd.concat([live_monthly.rename("monthly_return"), pct_ranks], axis=1))

            monthly_return  percentile_vs_historical
date                                                
2025-06-30          0.0120                    6.8966
2025-07-31          0.0318                   31.0345
2025-08-31          0.0395                   37.9310
2025-09-30          0.0247                   27.5862
2025-10-31          0.0839                   82.7586
2025-11-30         -0.0177                    0.0000
2025-12-31         -0.0113                    3.4483
2026-01-31          0.0498                   68.9655
2026-02-28          0.1467                  100.0000
2026-03-31          0.0516                   68.9655
2026-04-30          0.0580                   68.9655
2026-05-31         -0.0123                    0.0000
2026-06-30          0.1486                  100.0000
2026-07-31         -0.0281                    0.0000
2026-08-31         -0.0144                    0.0000
2026-09-30         -0.0706                    0.0000
2026-10-31         -0.0878                    

In [74]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=hist_monthly * 100, nbinsx=10, name="historical monthly returns",
                            marker_color="#636EFA", opacity=0.75))
for date, r in live_monthly.items():
    fig.add_vline(x=r * 100, line_color="#EF553B", line_width=2,
                  annotation_text=f"{date.strftime('%b %y')}: {pct_ranks[date]:.0f}th pct.",
                  annotation_position="top")
fig.update_layout(title="Live monthly returns vs. historical distribution", template=TEMPLATE,
                   xaxis_title="monthly return (%)", yaxis_title="count", height = 1000)
fig.show()

## 5. CUSUM

Sequential mean-shift detector — accumulates standardized deviations
from the historical mean, so it catches a small but *persistent* shift
faster than "wait N months below a band" would:

$$z(t) = \frac{r(t) - \mu_0}{\sigma_0}, \qquad
S^+(t) = \max\big(0,\; S^+(t{-}1) + z(t) - k\big), \qquad
S^-(t) = \min\big(0,\; S^-(t{-}1) + z(t) + k\big)$$

Alarm when $S^+(t) > h$ or $S^-(t) < -h$. $k$ (slack) absorbs normal
noise — only a sustained shift accumulates past it. $h$ (threshold) is
calibrated via the same block bootstrap as section 2, targeting a
chosen false-alarm rate, rather than picked as a generic "5 sigma" rule.

In [75]:
def cusum(returns, mu0, sigma0, k_mult=0.5, h_mult=5.0):
    z = (np.asarray(returns) - mu0) / sigma0
    n = len(z)
    s_pos, s_neg = np.zeros(n), np.zeros(n)
    alarms = []
    for t in range(1, n):
        s_pos[t] = max(0.0, s_pos[t - 1] + z[t] - k_mult)
        s_neg[t] = min(0.0, s_neg[t - 1] + z[t] + k_mult)
        if s_pos[t] > h_mult or s_neg[t] < -h_mult:
            alarms.append(t)
    return pd.DataFrame({"z": z, "s_pos": s_pos, "s_neg": s_neg}), alarms


def calibrate_cusum_h(wf_returns, k_mult, target_far, horizon, block_size, n_sims, seed=0):
    """Bootstrap the false-alarm rate under H0 (no real shift) to pick h."""
    mu0, sigma0 = wf_returns.mean(), wf_returns.std()
    sims = block_bootstrap_paths(wf_returns, n_sims, horizon, block_size, seed=seed)
    peak = np.empty(n_sims)
    for i in range(n_sims):
        z = (sims[i] - mu0) / sigma0
        s_pos = s_neg = worst = 0.0
        for zt in z:
            s_pos = max(0.0, s_pos + zt - k_mult)
            s_neg = min(0.0, s_neg + zt + k_mult)
            worst = max(worst, s_pos, -s_neg)
        peak[i] = worst
    return float(np.quantile(peak, 1 - target_far))


mu0, sigma0 = hist_df["return"].mean(), hist_df["return"].std()
h = calibrate_cusum_h(hist_df["return"].values, k_mult=0.5, target_far=0.05,
                       horizon=N_LIVE, block_size=block_size, n_sims=1000, seed=2)
cusum_df, alarms = cusum(live_df["return"].values, mu0, sigma0, k_mult=0.5, h_mult=h)
print(f"calibrated threshold h = {h:.2f}   |   first alarm at live trade #: {alarms[0] if alarms else None}")

calibrated threshold h = 8.42   |   first alarm at live trade #: 322


In [76]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=cusum_df["s_pos"], mode="lines", name="S+ (shift up)", line=dict(color="#00CC96")))
fig.add_trace(go.Scatter(y=cusum_df["s_neg"], mode="lines", name="S- (shift down)", line=dict(color="#EF553B")))
fig.add_hline(y=h, line_dash="dash", line_color="gray")
fig.add_hline(y=-h, line_dash="dash", line_color="gray", annotation_text=f"threshold +/-{h:.2f}")
for t in alarms:
    fig.add_vline(x=t, line_dash="dot", line_color="red", opacity=0.3)
fig.update_layout(title="CUSUM — live returns vs. historical benchmark", template=TEMPLATE,
                   xaxis_title="trade #", yaxis_title="CUSUM statistic", height = 1000)
fig.show()

## 6. Rolling alpha regression

Same benchmark, tested as a formal rolling test instead of a sequential
one — OLS with an intercept only, HC3-robust standard errors (robust to
changing volatility, unlike a naive t-test):

$$r_{\text{live}}(t) - \mu_0 = \alpha + \varepsilon(t)$$

$\alpha$ is the window's mean live return minus the historical
benchmark; a significant decay signal is when the **upper bound of the
95% CI drops below zero** — the underperformance in that window can't
be explained by sampling noise alone.

The window length is not arbitrary either: it is sized from the
strategy's trading frequency and the smallest shift worth detecting,
via a standard power calculation.

In [77]:
def recommended_window(trades_per_year, detectable_shift_sigma=0.5, alpha=0.05, power=0.8):
    z_alpha = norm.ppf(1 - alpha / 2)
    z_power = norm.ppf(power)
    n = int(np.ceil(((z_alpha + z_power) / detectable_shift_sigma) ** 2))
    return n, n / trades_per_year * 12   # (window in trades, approx. delay in months)


def rolling_alpha(returns, mu0, window):
    returns = np.asarray(returns)
    rows = []
    for t in range(window, len(returns) + 1):
        y = returns[t - window:t] - mu0
        X = np.ones((window, 1))
        fit = sm.OLS(y, X).fit(cov_type="HC3")
        ci_lo, ci_hi = fit.conf_int()[0]
        rows.append({"t": t, "alpha": fit.params[0], "ci_low": ci_lo, "ci_high": ci_hi, "pvalue": fit.pvalues[0]})
    return pd.DataFrame(rows)


window, delay_months = recommended_window(TRADES_PER_YEAR, detectable_shift_sigma=0.5)
print(f"window = {window} trades  (~{delay_months:.1f} months of detection delay)")

alpha_df = rolling_alpha(live_df["return"].values, mu0, window)
n_sig = (alpha_df["ci_high"] < 0).sum()
print(f"significant decay windows: {n_sig} / {len(alpha_df)}")

window = 32 trades  (~1.5 months of detection delay)
significant decay windows: 70 / 319


In [78]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=alpha_df["t"], y=alpha_df["ci_high"], mode="lines",
                          line=dict(color="rgba(99,110,250,0)"), showlegend=False))
fig.add_trace(go.Scatter(x=alpha_df["t"], y=alpha_df["ci_low"], mode="lines",
                          line=dict(color="rgba(99,110,250,0)"), fill="tonexty",
                          fillcolor="rgba(99,110,250,0.2)", name="95% CI (HC3)"))
fig.add_trace(go.Scatter(x=alpha_df["t"], y=alpha_df["alpha"], mode="lines",
                          line=dict(color="#636EFA"), name="alpha (live mean - mu0)"))
fig.add_hline(y=0, line_color="white", line_width=1)
sig = alpha_df["ci_high"] < 0
fig.add_trace(go.Scatter(x=alpha_df.loc[sig, "t"], y=alpha_df.loc[sig, "alpha"], mode="markers",
                          marker=dict(color="red", size=7), name="significant decay"))
fig.update_layout(title="Rolling alpha regression — live vs. historical benchmark", template=TEMPLATE,
                   xaxis_title="trade # (window end)", yaxis_title="alpha vs. benchmark", height = 1000)
fig.show()

## Takeaway

All three independent signals — the pre-committed drawdown curve, CUSUM,
and the rolling alpha regression — agree on roughly where the engineered
decay starts, and none of them needed a single number decided *after*
looking at the live data. That agreement across independent methods is
the actual bar for calling something a real decay, not any single chart
crossing a line.